Transcriptor factor evaluation

In [1]:
import scanpy as sc
import decoupler as dc

# Only needed for processing
import numpy as np
import pandas as pd
from anndata import AnnData

In [2]:
import pandas as pd
import yaml

In [6]:
dds_dict = {'ridge':'../data/feature_importance_ridge_smote_7b_SYMBOLS.csv',
           'catboost':'../data/feature_importance_smote_catboost_7b_gene_ranking_07.csv'}

In [7]:
file = dds_dict['catboost']
ridge = pd.read_csv(file, index_col=0)
ridge_

,Ensembl,coef,abs_coef
Symbol,,,
NT5C2,ENSG00000076685.18,1.413563e+01,1.413563e+01
AC012146.4,ENSG00000262855.1,1.204186e+01,1.204186e+01
UBFD1,ENSG00000103353.15,1.083837e+01,1.083837e+01
ALDOA,ENSG00000285043.1,7.641741e+00,7.641741e+00
H3F3B,ENSG00000132475.10,6.241443e+00,6.241443e+00
...,...,...,...
RSPRY1,ENSG00000159579.13,5.070000e-06,5.070000e-06
RNA5SP385,ENSG00000251756.1,2.550000e-06,2.550000e-06
TRIM24,ENSG00000122779.17,8.830000e-07,8.830000e-07


In [8]:
mat_dict = {}
for dds, file in dds_dict.items():
    column_name = ''
    if dds == 'ridge':
        column_name= 'Importance'
    elif dds=='catboost':
        column_name='abs_coef'
    mat = pd.read_csv(file, index_col=0)[[column_name]].T.rename(index={column_name: dds})
    mat.T.to_csv(f'results/mat_{dds}.csv')
    mat_dict[dds]=mat.copy()

In [9]:
collectri = pd.read_csv('collectri.csv')

In [10]:
len(collectri['target'].unique())

6692

In [11]:
mat_dict['ridge']

,TSPAN6,TNMD,DPM1,SCYL3,FIRRM,FGR,CFH,FUCA2,GCLC,NFYA,...,NaN,C4orf36,TUSC2P1,NaN,OR4M2-OT1,H2BK1,OR1Q1BP,NaN,NaN,TBCEL-TECTA
ridge,0.121967,0.054519,0.084876,0.042627,0.002908,0.072843,-0.012397,-0.085086,-0.102701,-0.009736,...,0.068401,-0.079764,0.061651,0.052366,0.060823,0.035708,0.063308,0.060353,-0.004797,0.066252


In [12]:
test = 'MTSS1'
collectri[(collectri['source']==test) | (collectri['target']==test)]

,Unnamed: 0,source,target,weight,PMID
21037,21037,DNMT3B,MTSS1,1,21909138
29540,29540,NR1I2,MTSS1,1,19129222
36480,36480,TBX5,MTSS1,1,20802524


In [14]:
mat.columns[mat.columns.duplicated()].unique()


Index([nan, 'PINX1', 'SIGLEC5', 'MATR3', 'PDE8B', 'PDE4C', 'POLR2J3',
       'C4orf36'],
      dtype='object')

In [15]:
# Infer TF activities with ulm
tf_dict = {}
for comparison, mat in mat_dict.items():
    comparison = comparison[:-5]
    mat = mat.loc[:, ~mat.columns.duplicated()]
    tf_dict[comparison] = dc.run_ulm(mat=mat, net=collectri, verbose=True) # tf_acts, tf_pvals
    tf_acts=tf_dict[comparison][0]
    tf_pvals = tf_dict[comparison][1]
    tf_df = pd.DataFrame(tf_acts.T)
    tf_df['pvals']=tf_pvals.T
    tf_df.to_csv(f'results/tf_acts_{comparison}.csv')
    tf_df

Running ulm on mat with 1 samples and 29303 targets for 773 sources.
Running ulm on mat with 1 samples and 354 targets for 63 sources.


In [18]:
tf_dict

{'': (           ABL1      AHR      AHRR       AIP      AIRE     APEX1        AR  \
  ridge  1.630651  0.07347 -0.435519  0.203335  0.338418  0.270321  0.257853   
  
           ARID1A    ARID1B    ARID3A  ...    ZNF382    ZNF384    ZNF395  \
  ridge -0.898728  0.067883 -0.003524  ...  0.349349 -0.268506 -0.315532   
  
           ZNF410    ZNF436    ZNF699     ZNF76   ZNF804A     ZNF91      ZXDC  
  ridge -1.022144 -1.017652 -0.782328 -1.014318 -0.870431  1.090518 -0.132898  
  
  [1 rows x 773 columns],
             ABL1       AHR      AHRR       AIP      AIRE     APEX1        AR  \
  ridge  0.102975  0.941432  0.663189  0.838875  0.735051  0.786915  0.796522   
  
           ARID1A    ARID1B    ARID3A  ...   ZNF382    ZNF384   ZNF395  \
  ridge  0.368805  0.945879  0.997188  ...  0.72683  0.788312  0.75236   
  
           ZNF410    ZNF436    ZNF699    ZNF76   ZNF804A     ZNF91      ZXDC  
  ridge  0.306721  0.308852  0.434028  0.31044  0.384072  0.275494  0.894275  
  
  [1 rows x 

In [17]:
tf_dict['catboost'][0]

KeyError: 'catboost'

In [ ]:
tf_dict['yo_file'][1]